# Prometheus-1B: Phase 3 - Supervised Fine-Tuning (SFT)
**CPT 완료 모델에 지시 수행(Instruction Following) 능력 부여**

CPT로 지식을 주입한 모델은 아직 "대화"를 할 줄 모릅니다.  
SFT 단계에서 고품질 지시-응답 데이터로 학습하여 실제 사용 가능한 모델로 만듭니다.

| 항목 | 설정 |
|------|------|
| 방식 | LoRA (r=64, alpha=128) - 효율적 파인튜닝 |
| 데이터 | OpenHermes 2.5 + MetaMathQA + CodeAlpaca |
| Epochs | 3 |
| LR | 1e-4 |

**예상 소요**: 3-5시간 (A100)

## 1. 환경 설정

In [ ]:
%%capture
# TRL 제거 - 순수 transformers + peft 로만 학습 (버전 변경 에러 원천 차단)
!pip install transformers datasets accelerate peft bitsandbytes

In [ ]:
import os
import torch
from google.colab import drive, userdata

drive.mount('/content/drive')
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

CPT_MODEL_DIR = "/content/drive/MyDrive/Prometheus-1B-CPT-Final"
SFT_OUTPUT_DIR = "/content/drive/MyDrive/Prometheus-1B-SFT"
FINAL_MERGED_DIR = "/content/drive/MyDrive/Prometheus-1B-Final"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")
print(f"CPT model: {CPT_MODEL_DIR}")
assert os.path.exists(CPT_MODEL_DIR), f"CPT model not found at {CPT_MODEL_DIR}. Run Phase 2 first!"

In [ ]:
# Colab 세션 끊김 방지 (백그라운드 스레드)
import threading
import time
import IPython

def keep_alive():
    while True:
        time.sleep(120)  # 2분마다
        IPython.display.clear_output(wait=True)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("Keep-alive thread started (2min interval)")

## 2. CPT 모델 로드 + LoRA 적용

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MAX_SEQ_LENGTH = 4096

# CPT 완료 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(CPT_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA 적용
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# 학습 가능한 파라미터 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}%)")

## 3. SFT 데이터 준비
3가지 고품질 지시 데이터를 혼합합니다:
- **OpenHermes 2.5**: 범용 대화/지시 (MMLU, ARC 향상)
- **MetaMathQA**: 수학 문제 풀이 (GSM8K 향상)  
- **CodeAlpaca**: 코드 생성 (HumanEval 향상)

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Llama 3.2 Chat Template
CHAT_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are Prometheus, a helpful and knowledgeable AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>"""

def format_openhermes(example):
    """OpenHermes 2.5 포맷 변환"""
    convos = example.get("conversations", [])
    instruction = ""
    response = ""
    for msg in convos:
        if msg["from"] == "human":
            instruction = msg["value"]
        elif msg["from"] == "gpt":
            response = msg["value"]
    if instruction and response:
        return {"text": CHAT_TEMPLATE.format(instruction=instruction, response=response)}
    return {"text": ""}

def format_metamath(example):
    """MetaMathQA 포맷 변환"""
    return {"text": CHAT_TEMPLATE.format(
        instruction=example.get("query", ""),
        response=example.get("response", ""),
    )}

def format_codealpaca(example):
    """CodeAlpaca 포맷 변환"""
    instruction = example.get("instruction", "")
    inp = example.get("input", "")
    if inp:
        instruction = f"{instruction}\n\nInput:\n{inp}"
    return {"text": CHAT_TEMPLATE.format(
        instruction=instruction,
        response=example.get("output", ""),
    )}

# 데이터 로드 및 포맷팅
print("Loading OpenHermes 2.5...")
hermes = load_dataset("teknium/OpenHermes-2.5", split="train", token=hf_token)
hermes = hermes.shuffle(seed=42).select(range(min(100_000, len(hermes))))  # 10만개 샘플링
hermes = hermes.map(format_openhermes, remove_columns=hermes.column_names)
hermes = hermes.filter(lambda x: len(x["text"]) > 100)

print("Loading MetaMathQA...")
math_data = load_dataset("meta-math/MetaMathQA", split="train", token=hf_token)
math_data = math_data.shuffle(seed=42).select(range(min(50_000, len(math_data))))  # 5만개
math_data = math_data.map(format_metamath, remove_columns=math_data.column_names)

print("Loading CodeAlpaca...")
code_data = load_dataset("sahil2801/CodeAlpaca-20k", split="train", token=hf_token)
code_data = code_data.map(format_codealpaca, remove_columns=code_data.column_names)

# 합치기 + 셔플
sft_dataset = concatenate_datasets([hermes, math_data, code_data])
sft_dataset = sft_dataset.shuffle(seed=42)

# Train/Eval 분할
sft_split = sft_dataset.train_test_split(test_size=0.02, seed=42)
sft_train = sft_split["train"]
sft_eval = sft_split["test"]

print(f"\nSFT Dataset:")
print(f"  Train: {len(sft_train):,} examples")
print(f"  Eval:  {len(sft_eval):,} examples")
print(f"  Sample: {sft_train[0]['text'][:300]}...")

## 4. SFT 학습

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 데이터 토크나이징
def tokenize_fn(examples):
    # labels는 DataCollatorForLanguageModeling이 자동 생성 - 직접 넣으면 충돌
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,  # 배치 내 동적 패딩은 collator가 처리
    )

print("Tokenizing train dataset...")
train_tokenized = sft_train.map(
    tokenize_fn, batched=True, remove_columns=["text"],
    num_proc=2, desc="Tokenizing train"
)
print("Tokenizing eval dataset...")
eval_tokenized = sft_eval.map(
    tokenize_fn, batched=True, remove_columns=["text"],
    num_proc=2, desc="Tokenizing eval"
)

# mlm=False → input_ids를 그대로 labels로 복사 (CLM 방식)
# pad_to_multiple_of=8 → Tensor Core 활용 효율 향상
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    weight_decay=0.01,
    bf16=True,
    num_train_epochs=3,
    save_steps=500,
    save_total_limit=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    report_to="none",
    gradient_checkpointing=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

# 체크포인트 자동 감지
sft_checkpoint = None
if os.path.isdir(SFT_OUTPUT_DIR):
    checkpoints = [
        os.path.join(SFT_OUTPUT_DIR, d) for d in os.listdir(SFT_OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        sft_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"Resuming SFT from checkpoint: {sft_checkpoint}")
    else:
        print("No checkpoint found. Starting fresh SFT training.")
else:
    print("Starting fresh SFT training.")

print("Starting SFT training...")
trainer.train(resume_from_checkpoint=sft_checkpoint)

## 5. LoRA 가중치 병합 및 최종 모델 저장
LoRA 가중치를 Base 모델에 병합하여 독립적으로 사용 가능한 최종 모델을 만듭니다.

In [ ]:
from peft import PeftModel

# 최신 체크포인트 자동 탐색 (SFT_OUTPUT_DIR 루트가 아닌 checkpoint 폴더 안에 LoRA 가중치 저장됨)
checkpoints = [
    os.path.join(SFT_OUTPUT_DIR, d) for d in os.listdir(SFT_OUTPUT_DIR)
    if d.startswith("checkpoint-")
]
assert checkpoints, f"No checkpoints found in {SFT_OUTPUT_DIR}"
latest_ckpt = max(checkpoints, key=os.path.getmtime)
print(f"Using checkpoint for merge: {latest_ckpt}")

# LoRA 가중치를 원본 모델에 병합
print("Merging LoRA weights into base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
merged_model = PeftModel.from_pretrained(base_model, latest_ckpt)
merged_model = merged_model.merge_and_unload()

# 최종 모델 저장
os.makedirs(FINAL_MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(FINAL_MERGED_DIR)
tokenizer.save_pretrained(FINAL_MERGED_DIR)

print(f"Final merged model saved to: {FINAL_MERGED_DIR}")
print(f"Model size: {sum(p.numel() for p in merged_model.parameters()) / 1e9:.2f}B parameters")

## 6. 빠른 테스트
모델이 제대로 대화하는지 간단히 확인합니다.

In [ ]:
from peft import PeftModel

# 최신 체크포인트로 병합 (이미 위에서 merged_model이 있으면 재사용 가능)
if 'merged_model' not in dir():
    checkpoints = [
        os.path.join(SFT_OUTPUT_DIR, d) for d in os.listdir(SFT_OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    latest_ckpt = max(checkpoints, key=os.path.getmtime)
    print(f"Loading from checkpoint: {latest_ckpt}")
    base_model = AutoModelForCausalLM.from_pretrained(
        CPT_MODEL_DIR, torch_dtype=torch.bfloat16, device_map="auto",
    )
    merged_model = PeftModel.from_pretrained(base_model, latest_ckpt)
    merged_model = merged_model.merge_and_unload()

# 추론 테스트
test_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a Python function that checks if a number is prime.",
    "What is 15% of 240?",
]

for prompt in test_prompts:
    formatted = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        "You are a helpful and knowledgeable AI assistant.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response[:500]}")

print(f"\n{'='*60}")
print("SFT complete! Proceed to Phase 4 (Evaluation) with notebook 04_evaluate.ipynb")